# AI Agent Security — Multi-Step Tool Attacks

Competition submission for **ai-agent-security-multi-step-tool-attacks**.

Writes `attack.py` (defining `AttackAlgorithm(AttackAlgorithmBase)`) to `/kaggle/working/`. On the competition rerun the evaluation server scores it against GPT-OSS and Gemma 4; during Save & Run All a placeholder `submission.csv` is written so the notebook is eligible to submit.

In [ ]:
import sys, glob, os
from pathlib import Path

sys.argv = [sys.argv[0]]  # avoid argparse conflicts in Kaggle

# The competition dataset ships kaggle_evaluation/ and aicomp_sdk/ at its root.
dataset_root = None
for candidate in glob.glob('/kaggle/input/**/kaggle_evaluation', recursive=True):
    dataset_root = str(Path(candidate).parent)
    if dataset_root not in sys.path:
        sys.path.insert(0, dataset_root)
    print('Dataset root:', dataset_root)
    break
assert dataset_root, 'competition dataset (kaggle_evaluation/) not found under /kaggle/input'
import aicomp_sdk  # noqa: F401
print('Setup complete; aicomp_sdk importable')

In [ ]:
import base64, importlib.util
from pathlib import Path

ATTACK_PY_B64 = '''\
...
'''

attack_path = Path('/kaggle/working/attack.py')
attack_path.write_bytes(base64.b64decode(ATTACK_PY_B64))
print('Wrote', attack_path, 'bytes=', attack_path.stat().st_size)

# Sanity-check the written module defines AttackAlgorithm.
spec = importlib.util.spec_from_file_location('attack_submission', attack_path)
mod = importlib.util.module_from_spec(spec)
assert spec and spec.loader
spec.loader.exec_module(mod)
assert hasattr(mod, 'AttackAlgorithm'), 'AttackAlgorithm missing from attack.py'
print('AttackAlgorithm importable:', mod.AttackAlgorithm)


In [ ]:
import os
from pathlib import Path

SUBMISSION_PATH = Path('/kaggle/working/submission.csv')

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Real scoring: the gateway drives AttackAlgorithm.run against the target models.
    import kaggle_evaluation.jed_attack_134815.jed_attack_inference_server as server
    server.JEDAttackInferenceServer().serve()
else:
    # Save & Run All: write the required placeholder so the notebook can be submitted.
    # The competition rerun overwrites this with real scores.
    SUBMISSION_PATH.write_text(
        'Id,Score\n'
        'gpt_oss_public,0.0\n'
        'gpt_oss_private,0.0\n'
        'gemma_public,0.0\n'
        'gemma_private,0.0\n',
        encoding='utf-8',
    )
    print('Wrote placeholder', SUBMISSION_PATH)